#####Projeto Classificação de Grãos de Trigo (Seeds Dataset): em cooperativas agrícolas menores, a classificação dos grãos é feita de forma manual sujeito a demora e a erros humanos. O objetivo deste exercício é aplicar a metodologia CRISP-DM para desenvolver um modelo de aprendizado de máquina que classifique variedades de grãos de trigo com base em suas características físicas. O conjunto de dados contém medições de 210 amostras de grãos de trigo pertencentes a três variedades diferentes:Kama, Rosa e a Canadian.

#### Análise e pré-processamento dos dados fornecidos


In [ ]:
# Arquivo Seeds_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Nomeação das colunas
nomes_colunas = [
    'Area', 'Perimetro', 'Compacidade', 'Comprimento_Grao',
    'Largura_Grao', 'Assimetria', 'Comprimento_Sulco', 'Classe'
]

# Carregar o arquivo
df = pd.read_csv(
    'seeds_dataset.csv',
    sep=r'\s+',
    names=nomes_colunas,
    header=None
)

df.head()

# Nomes científicos das sementes
mapeamento_classes = {
    1: 'Kama',
    2: 'Rosa',
    3: 'Canadian'
}

df['Classe'] = df['Classe'].map(mapeamento_classes)
df.head()

print("--- Estrutura dos Dados ---")
df.info()


#### Estatísticas descritivas


In [2]:
# Média, Mediana/50%, Desvio Padrão/std
print("\n--- Estatísticas Descritivas ---")
df.describe(include='all').T


--- Estatísticas Descritivas ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Area,210.0,NaN,NaN,NaN,14.847524,2.909699,10.59,12.27,14.355,17.305,21.18
Perimetro,210.0,NaN,NaN,NaN,14.559286,1.305959,12.41,13.45,14.32,15.715,17.25
Compacidade,210.0,NaN,NaN,NaN,0.870999,0.023629,0.8081,0.8569,0.87345,0.887775,0.9183
Comprimento_Grao,210.0,NaN,NaN,NaN,5.628533,0.443063,4.899,5.26225,5.5235,5.97975,6.675
Largura_Grao,210.0,NaN,NaN,NaN,3.258605,0.377714,2.63,2.944,3.237,3.56175,4.033
Assimetria,210.0,NaN,NaN,NaN,3.700201,1.503557,0.7651,2.5615,3.599,4.76875,8.456
Comprimento_Sulco,210.0,NaN,NaN,NaN,5.408071,0.49148,4.519,5.045,5.223,5.877,6.55
Classe,210,3,Kama,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Visualização dos Gráficos


In [ ]:
# Gráficos de Histograma, Boxplot, Dispersão e Matriz de correlação
sns.set_theme(style="whitegrid")

num_cols = df.select_dtypes(include=[np.number]).columns
features = [col for col in num_cols if col != 'Classe']

# Histogramas coloridos por classe de grão
for col in features:
    plt.figure(figsize=(8, 4))
    sns.histplot(
        data=df, x=col, hue='Classe',
        kde=True, bins=25, palette='Set2', multiple='stack'
    )
    plt.title(f'Distribuição de {col} por Classe')
    plt.show()

# Boxplots com cor distinta por característica
cores = sns.color_palette("deep", len(features))

for i, col in enumerate(features):
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col], color=cores[i])
    plt.title(f'Boxplot de {col}')
    plt.show()

# Gráfico de dispersão geral (pairplot)
sns.pairplot(df, hue='Classe', palette='Set2', corner=True)
plt.suptitle('Gráfico de Dispersão Geral (Pairplot)', y=1.02)
plt.show()

# Matriz de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[num_cols].corr(), annot=True,
    cmap='RdYlBu_r', fmt=".2f",
    vmin=-1, vmax=1, linewidths=0.5
)
plt.title('Matriz de Correlação')
plt.show()


#### Identificação dos valores ausentes

In [ ]:
# Checagem de valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())

# Preenchimento com a mediana onde houver nulos
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())


#### Padronização das características


In [ ]:
# Padronização: média = 0, desvio padrão = 1
colunas_numericas = df.drop(columns=['Classe']).columns
scaler = StandardScaler()
df_processado = df.copy()
df_processado[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])

print("--- DataFrame com Dados Padronizados e Classe Integrada ---")
df_processado.head()


#### Implementação e comparação dos diferentes algoritmos de classificação


In [ ]:
# Definição dos algoritmos de classificação: KNN, SVM e Random Forest
modelos = {
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

resultados_acuracia = {}


#### Divisão em treino e teste


In [ ]:
# Divisão treino/teste com estratificação por classe
X = df_processado.drop(columns=['Classe'])
y = df_processado['Classe']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Dados de Treinamento: {X_train.shape[0]} amostras")
print(f"Dados de Teste:       {X_test.shape[0]} amostras")


#### Definição dos Modelos


In [55]:
# KNN, SVM e Random Forest
modelos = {
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Dicionário para armazenar a acurácia de cada um para comparação posterior
resultados_acuracia = {}


#### Treinamento, predição e avaliação


In [ ]:
# Treinamento, predição e avaliação: acurácia, F1, precisão, recall e support
for nome, modelo in modelos.items():
    print(f"=== Treinando o modelo: {nome} ===")

    # Treinar o modelo
    modelo.fit(X_train, y_train)

    # Predições no conjunto de teste
    y_pred = modelo.predict(X_test)

    # Acurácia geral
    acuracia = accuracy_score(y_test, y_pred)
    resultados_acuracia[nome] = acuracia
    print(f"Acurácia Geral: {acuracia:.4f}")

    # Relatório detalhado por classe
    print("\nRelatório de Classificação detalhado:")
    print(classification_report(y_test, y_pred))

    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred, labels=['Kama', 'Rosa', 'Canadian'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Kama', 'Rosa', 'Canadian'],
        yticklabels=['Kama', 'Rosa', 'Canadian']
    )
    plt.title(f'Matriz de Confusão - {nome}')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.show()
    print("-" * 60 + "\n")


#### Comparação dos algoritmos

In [ ]:
# Desempenho final
print("=== COMPARAÇÃO FINAL DE DESEMPENHO ===")
for nome, acc in resultados_acuracia.items():
    print(f"-> {nome}: {acc * 100:.2f}% de Acurácia")


#### Otimização dos modelos para melhorar o desempenho


In [ ]:
# Otimização dos modelos via Grid Search
print("=== INICIANDO A OTIMIZAÇÃO DE HIPERPARÂMETROS (GRID SEARCH) ===\n")

# Grades de parâmetros para cada modelo
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

config_otimizacao = {
    "K-Nearest Neighbors (KNN)": (KNeighborsClassifier(), param_grid_knn),
    "Support Vector Machine (SVM)": (SVC(random_state=42), param_grid_svm),
    "Random Forest": (RandomForestClassifier(random_state=42), param_grid_rf)
}

resultados_otimizados = {}


#### Execução do GRID SEARCH e reavaliação


In [ ]:
# Execução do Grid Search e reavaliação dos modelos
for nome, (modelo_base, grade) in config_otimizacao.items():
    print(f"Otimizando {nome}...")

    grid_search = GridSearchCV(
        estimator=modelo_base, param_grid=grade,
        cv=5, scoring='accuracy', n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    melhor_modelo = grid_search.best_estimator_
    print(f"-> Melhores parâmetros encontrados: {grid_search.best_params_}")

    y_pred_otimizado = melhor_modelo.predict(X_test)

    acc_otimizada = accuracy_score(y_test, y_pred_otimizado)
    resultados_otimizados[nome] = acc_otimizada
    print(f"Nova Acurácia Geral: {acc_otimizada:.4f}")

    print("\nNovo Relatório de Classificação:")
    print(classification_report(y_test, y_pred_otimizado))

    cm_otimizada = confusion_matrix(
        y_test, y_pred_otimizado, labels=['Kama', 'Rosa', 'Canadian']
    )
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm_otimizada, annot=True, fmt='d', cmap='Greens',
        xticklabels=['Kama', 'Rosa', 'Canadian'],
        yticklabels=['Kama', 'Rosa', 'Canadian']
    )
    plt.title(f'Matriz Otimizada - {nome}')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.show()
    print("-" * 60 + "\n")


#### Comparação antes versus depois

In [ ]:
# Comparação antes vs. depois da otimização
print("=== COMPARAÇÃO DE MELHORIA DO DESEMPENHO ===")
for nome in modelos.keys():
    acc_anterior = resultados_acuracia[nome] * 100
    acc_nova = resultados_otimizados[nome] * 100
    ganho = acc_nova - acc_anterior

    print(f"-> {nome}:")
    print(f"   Acurácia Inicial:   {acc_anterior:.2f}%")
    print(f"   Acurácia Otimizada: {acc_nova:.2f}%")
    if ganho > 0:
        print(f"   Melhoria significativa de +{ganho:.2f}%!")
    elif ganho == 0:
        print("   Manteve o excelente desempenho original (já estava no limite máximo).")
    else:
        print(f"   Variação marginal de {ganho:.2f}%.")


#### 🏆 Conclusão: Qual modelo se saiu melhor?


In [ ]:
# Interpretação dos Resultados e Insights Relevantes

Nesta etapa final, analisamos como os modelos matemáticos se comportaram diante das características físicas das sementes e o que isso significa no contexto da classificação de grãos.

### 📊 Retrospectiva do Desempenho (Placar Geral)
* **KNN:** 88.89% de acurácia
* **SVM:** 88.89% de acurácia
* **Random Forest:** 84.13% de acurácia

---

### 🔍 Análise Profunda e Insights do Contexto dos Grãos

#### 1. Por que o KNN e o SVM foram os melhores? (A Geometria dos Grãos)
O empate do **KNN** e do **SVM** com **88.89%** de acertos tem uma explicação prática ligada à natureza dos dados. Os grãos de trigo diferem entre si por medidas contínuas e proporcionais (um grão maior geralmente tem um perímetro maior e um sulco mais longo).No caso das espécies estudadas no nosso dataset, a canadian é um grão pequeno, a karma um grão médio e a rosa um grão do tipo longo.
No início do projeto, quando aplicamos a **Padronização**, nós colocamos todas essas medidas na mesma escala espacial. Como o KNN decide com base na proximidade física dos pontos e o SVM cria linhas de separação geométricas perfeitas, eles conseguiram agrupar com muita facilidade as sementes que tinham formatos parecidos. O sucesso deles prova que a nossa etapa de pré-processamento de dados foi certeira.

#### 2. O comportamento da Random Forest (A fraqueza das regras rígidas)
A **Random Forest** pontuou menos (**84.13%**). Esse algoritmo funciona criando ramificações de "sim ou não" baseadas em cortes fixos (ex: *"A área é maior que 15?"*).
Na classificação de grãos, a transição de tamanho entre uma variedade e outra não é tão rígida; existe uma área de transição onde sementes grandes de um tipo podem ter o mesmo tamanho de sementes médias de outro. Como a Random Forest tenta criar caixas rígidas para separar os dados, ela acabou errando mais nessa zona de transição, mostrando que modelos puramente geométricos se adaptam melhor a este problema.

#### 3. O limite da Otimização e o "Mistério" dos 11% de Erros
A nossa busca por melhores parâmetros (Grid Search) não alterou as notas dos modelos. Ao olhar de perto as **Matrizes de Confusão** e os nossos **Histogramas** iniciais, descobrimos o motivo:
* A variedade **Rosa** possui grãos gigantescos e isolados. Por isso, os modelos praticamente não erram o trigo Rosa.
* O verdadeiro desafio está entre as variedades **Kama** e **Canadian**. Nos gráficos de distribuição, vimos que a Área, o Perímetro e o Comprimento dessas duas sementes se sobrepõem quase por completo.

Na prática, isso significa que cerca de 11% das sementes *Kama* e *Canadian* são geometricamente idênticas por fora. Nenhum algoritmo de Machine Learning conseguirá atingir 100% de acerto usando apenas essas medidas, pois a resposta para diferenciá-las completamente exigiria dados complementares que não temos no arquivo, como o peso interno ou uma análise de DNA do grão.

---

### 🏁 Conclusão do Pipeline
O projeto atingiu seu objetivo de ponta a ponta. Conseguimos criar uma inteligência artificial capaz de identificar o tipo de trigo com quase **90% de precisão**. Para um sistema automatizado de triagem de grãos em uma fazenda ou cooperativa, o uso do modelo **SVM** ou **KNN** já traria uma economia enorme de tempo, substituindo a análise manual humana por uma checagem de frações de segundo.

